# LeetCode 1391 — Check if There is a Valid Path in a Grid bằng Python

Link đề: https://leetcode.com/problems/check-if-there-is-a-valid-path-in-a-grid/ · Độ khó: **Medium** · Chủ đề: Array, Depth-First Search, Breadth-First Search, Union-Find, Matrix

Skeleton LeetCode:

```python
class Solution:
    def hasValidPath(self, grid: list[list[int]]) -> bool:
        ...
```

## 1. Mục tiêu học

1. Biết **mô hình hoá lưới thành đồ thị**: mỗi ô là một đỉnh, hai ô kề nhau có cạnh **chỉ khi cả hai đều mở cửa về phía nhau**.
2. Viết được **bảng hướng** (`dict` loại đường → danh sách hướng) và kỹ thuật **kiểm tra nối ngược** (ô kế phải có hướng ngược lại).
3. Cài đặt và so sánh 5 cách: phóng to lưới 3×3 (baseline), BFS, DFS bằng stack, Union-Find, và **đi theo đường** với bộ nhớ `O(1)`.
4. Hiểu vì sao mỗi ô có **đúng 2 cửa** khiến đường đi từ `(0, 0)` gần như "bị ép", và dùng điều đó để chứng minh tính đúng.
5. Tránh các lỗi kinh điển: quên kiểm tra nối ngược, quên `visited` (lặp vô hạn), đệ quy quá sâu trên lưới 300×300, nhầm chiều hàng/cột.

## 2. Đề bài & ví dụ

Cho lưới `m × n`. Mỗi ô `grid[i][j]` là một **đoạn đường** thuộc một trong 6 loại:

| Loại | Nối ô bên … với ô bên … | Hình |
|---|---|---|
| 1 | trái – phải | `─` |
| 2 | trên – dưới | `│` |
| 3 | trái – dưới | `┐` |
| 4 | phải – dưới | `┌` |
| 5 | trái – trên | `┘` |
| 6 | phải – trên | `└` |

Bắt đầu ở ô góc trên trái `(0, 0)`. Một **đường đi hợp lệ** bắt đầu ở `(0, 0)`, kết thúc ở góc dưới phải `(m − 1, n − 1)` và **chỉ đi theo đường phố** (không được xoay/đổi loại đường). Trả về `True` nếu tồn tại đường đi hợp lệ, ngược lại `False`.

**Ví dụ 1:** `grid = [[2,4,3],[6,5,2]]` → `True` (đi qua được mọi ô rồi tới đích).

**Ví dụ 2:** `grid = [[1,2,1],[1,2,1]]` → `False` (ô `(0,0)` loại 1 không nối được với ô nào khác).

**Ví dụ 3:** `grid = [[1,1,2]]` → `False` (kẹt ở `(0,1)`, không vào được `(0,2)` vì loại 2 không mở cửa bên trái).

**Ràng buộc:** `1 <= m, n <= 300`, `1 <= grid[i][j] <= 6`.

In [1]:
import random
import time
from collections import deque
from typing import List

examples = [
    ([[2, 4, 3], [6, 5, 2]], True),
    ([[1, 2, 1], [1, 2, 1]], False),
    ([[1, 1, 2]], False),
]
SHAPE = {1: "─", 2: "│", 3: "┐", 4: "┌", 5: "┘", 6: "└"}

def show(grid):
    for row in grid:
        print("  " + " ".join(SHAPE[v] for v in row) + "    " + str(row))

for grid, want in examples:
    show(grid)
    print("  ->", want)
    print()

  │ ┌ ┐    [2, 4, 3]
  └ ┘ │    [6, 5, 2]
  -> True

  ─ │ ─    [1, 2, 1]
  ─ │ ─    [1, 2, 1]
  -> False

  ─ ─ │    [1, 1, 2]
  -> False



## 3. Kiến thức nền

### 3.1 Toạ độ trên lưới và 4 hướng

Ô `(r, c)`: `r` là **hàng** (tăng khi đi xuống), `c` là **cột** (tăng khi sang phải). Bốn hướng biểu diễn bằng độ lệch `(dr, dc)`:

- lên `(-1, 0)`, xuống `(1, 0)`, trái `(0, -1)`, phải `(0, 1)`.

Lỗi hay gặp: nghĩ "lên" là `r + 1` như trục toạ độ toán học. Trên lưới máy tính, hàng 0 nằm **trên cùng**.

In [2]:
UP, DOWN, LEFT, RIGHT = (-1, 0), (1, 0), (0, -1), (0, 1)
r, c = 1, 1
for name, (dr, dc) in [("lên", UP), ("xuống", DOWN), ("trái", LEFT), ("phải", RIGHT)]:
    print(f"từ ({r},{c}) đi {name:5s} -> ({r + dr},{c + dc})")

từ (1,1) đi lên   -> (0,1)
từ (1,1) đi xuống -> (2,1)
từ (1,1) đi trái  -> (1,0)
từ (1,1) đi phải  -> (1,2)


### 3.2 Bảng hướng bằng `dict`

Thay vì viết 6 nhánh `if`, ta lưu **mỗi loại đường mở cửa về những hướng nào**. Tra bảng `O(1)` và code ngắn, ít sai.

In [3]:
DIRS = {
    1: [LEFT, RIGHT],
    2: [UP, DOWN],
    3: [LEFT, DOWN],
    4: [RIGHT, DOWN],
    5: [LEFT, UP],
    6: [RIGHT, UP],
}
for t, ds in DIRS.items():
    print(t, SHAPE[t], ds)

1 ─ [(0, -1), (0, 1)]
2 │ [(-1, 0), (1, 0)]
3 ┐ [(0, -1), (1, 0)]
4 ┌ [(0, 1), (1, 0)]
5 ┘ [(0, -1), (-1, 0)]
6 └ [(0, 1), (-1, 0)]


### 3.3 Hướng ngược: `(-dr, -dc)`

Đi từ A sang B theo hướng `d` thì B phải có cửa theo hướng **ngược** `(-dr, -dc)` để quay về A. Đây là chìa khoá của bài: hai ô **nối nhau** khi và chỉ khi A có `d` **và** B có `-d`.

In [4]:
def opposite(d):
    return (-d[0], -d[1])

print("ngược của phải:", opposite(RIGHT), "== trái?", opposite(RIGHT) == LEFT)
print("ngược của lên:", opposite(UP), "== xuống?", opposite(UP) == DOWN)
# Loại 1 (─) ở bên trái và loại 3 (┐) ở bên phải: 1 có RIGHT, 3 có LEFT -> nối
print("1 rồi 3 nối?", RIGHT in DIRS[1] and LEFT in DIRS[3])
# Loại 1 và loại 2 (│): 2 không có LEFT -> không nối
print("1 rồi 2 nối?", RIGHT in DIRS[1] and LEFT in DIRS[2])

ngược của phải: (0, -1) == trái? True
ngược của lên: (1, 0) == xuống? True
1 rồi 3 nối? True
1 rồi 2 nối? False


### 3.4 `deque` cho BFS và `list` làm stack cho DFS

- `deque.popleft()` lấy phần tử **đầu** trong `O(1)` → hàng đợi (BFS).
- `list.pop()` lấy phần tử **cuối** trong `O(1)` → ngăn xếp (DFS).
- Không dùng `list.pop(0)` cho hàng đợi: mỗi lần `O(n)`.

In [5]:
q = deque([(0, 0)])
q.append((0, 1)); q.append((1, 0))
print("BFS lấy:", q.popleft(), "| còn", list(q))
st = [(0, 0)]
st.append((0, 1)); st.append((1, 0))
print("DFS lấy:", st.pop(), "| còn", st)

BFS lấy: (0, 0) | còn [(0, 1), (1, 0)]
DFS lấy: (1, 0) | còn [(0, 0), (0, 1)]


### 3.5 Mảng `visited` 2 chiều

Tạo đúng cách: `[[False] * n for _ in range(m)]`. **Đừng** viết `[[False] * n] * m` — các hàng dùng chung một list, sửa một ô làm đổi cả cột.

In [6]:
m, n = 2, 3
bad = [[False] * n] * m
bad[0][1] = True
good = [[False] * n for _ in range(m)]
good[0][1] = True
print("sai :", bad)
print("đúng:", good)

sai : [[False, True, False], [False, True, False]]
đúng: [[False, True, False], [False, False, False]]


### 3.6 Union-Find (Disjoint Set Union) tóm tắt

Quản lý các nhóm rời nhau: `find(x)` trả về "đại diện" của nhóm chứa `x`, `union(a, b)` gộp hai nhóm. Với **nén đường đi** (path compression), mỗi thao tác gần như `O(1)`. Đánh số ô `(r, c)` thành `r * n + c`.

In [7]:
parent = list(range(6))
def find(x):
    while parent[x] != x:
        parent[x] = parent[parent[x]]   # nén đường đi (halving)
        x = parent[x]
    return x
def union(a, b):
    parent[find(a)] = find(b)

union(0, 1); union(1, 2); union(4, 5)
print("0 và 2 cùng nhóm?", find(0) == find(2))
print("0 và 4 cùng nhóm?", find(0) == find(4))

0 và 2 cùng nhóm? True
0 và 4 cùng nhóm? False


## 4. Ý tưởng nền tảng / trực giác

Tính tay **Ví dụ 1** `[[2,4,3],[6,5,2]]`:

```
│ ┌ ┐
└ ┘ │
```

- `(0,0)` loại 2 `│`: cửa **lên** (ra ngoài lưới) và **xuống** → `(1,0)`.
- `(1,0)` loại 6 `└`: có cửa **lên** (nối ngược với `(0,0)` ✔) và **phải** → `(1,1)`.
- `(1,1)` loại 5 `┘`: có cửa **trái** ✔ và **lên** → `(0,1)`.
- `(0,1)` loại 4 `┌`: có cửa **xuống** ✔ và **phải** → `(0,2)`.
- `(0,2)` loại 3 `┐`: có cửa **trái** ✔ và **xuống** → `(1,2)`.
- `(1,2)` loại 2 `│`: có cửa **lên** ✔ — đây là đích `(1,2)` ⇒ `True`.

**Ví dụ 3** `[[1,1,2]]` = `─ ─ │`: `(0,0) → (0,1)` được (1 có phải, 1 có trái). `(0,1) → (0,2)`: `(0,1)` có cửa phải nhưng `(0,2)` loại `│` **không có cửa trái** ⇒ không nối. Hết đường ⇒ `False`.

Nhận xét: **mỗi bước ta chỉ cần hai câu hỏi**: (1) ô hiện tại có cửa hướng `d`? (2) ô kế có cửa hướng `-d`?

In [8]:
def neighbors(grid, r, c):
    """Các ô kề NỐI THẬT SỰ với (r, c)."""
    m, n = len(grid), len(grid[0])
    out = []
    for dr, dc in DIRS[grid[r][c]]:
        nr, nc = r + dr, c + dc
        if 0 <= nr < m and 0 <= nc < n and (-dr, -dc) in DIRS[grid[nr][nc]]:
            out.append((nr, nc))
    return out

g = examples[0][0]
for r in range(2):
    for c in range(3):
        print((r, c), SHAPE[g[r][c]], "->", neighbors(g, r, c))

(0, 0) │ -> [(1, 0)]
(0, 1) ┌ -> [(0, 2), (1, 1)]
(0, 2) ┐ -> [(0, 1), (1, 2)]
(1, 0) └ -> [(1, 1), (0, 0)]
(1, 1) ┘ -> [(1, 0), (0, 1)]
(1, 2) │ -> [(0, 2)]


## 5. Cách 1 — Brute Force / baseline: phóng to lưới 3×3

### Ý tưởng

Cách "không cần suy luận nối ngược": vẽ mỗi ô thành một **ô vuông 3×3 pixel**. Pixel giữa luôn là đường; pixel ở cạnh nào thì tô nếu đường mở về phía đó. Ví dụ loại 1 `─`:

```
. . .
# # #
. . .
```

Khi ghép các ô cạnh nhau, hai pixel đường chỉ **chạm nhau** nếu cả hai ô đều tô pixel cạnh chung ⇒ phép kiểm tra nối ngược được làm **tự động** bởi hình học. Sau đó chỉ cần flood fill (BFS 4 hướng) trên lưới `3m × 3n` từ pixel giữa của `(0,0)` xem có tới pixel giữa của `(m−1, n−1)` không.

In [9]:
def upscale(grid):
    m, n = len(grid), len(grid[0])
    big = [[0] * (3 * n) for _ in range(3 * m)]
    for r in range(m):
        for c in range(n):
            cr, cc = 3 * r + 1, 3 * c + 1          # pixel giữa
            big[cr][cc] = 1
            for dr, dc in DIRS[grid[r][c]]:
                big[cr + dr][cc + dc] = 1          # pixel cạnh theo hướng mở
    return big

for row in upscale([[2, 4, 3], [6, 5, 2]]):
    print("".join("#" if x else "." for x in row))

.#.......
.#..####.
.#..#..#.
.#..#..#.
.####..#.
.......#.


In [10]:
def valid_path_upscale(grid):
    m, n = len(grid), len(grid[0])
    big = upscale(grid)
    H, W = 3 * m, 3 * n
    start, goal = (1, 1), (3 * m - 2, 3 * n - 2)
    seen = [[False] * W for _ in range(H)]
    seen[1][1] = True
    q = deque([start])
    while q:
        r, c = q.popleft()
        if (r, c) == goal:
            return True
        for dr, dc in (UP, DOWN, LEFT, RIGHT):
            nr, nc = r + dr, c + dc
            if 0 <= nr < H and 0 <= nc < W and big[nr][nc] and not seen[nr][nc]:
                seen[nr][nc] = True
                q.append((nr, nc))
    return False

for grid, want in examples:
    print(grid, "->", valid_path_upscale(grid), "| mong đợi", want)

[[2, 4, 3], [6, 5, 2]] -> True | mong đợi True
[[1, 2, 1], [1, 2, 1]] -> False | mong đợi False
[[1, 1, 2]] -> False | mong đợi False


### Độ phức tạp và nút thắt

- Time `O(9·m·n)` = `O(m·n)`, Space `O(9·m·n)`.
- Đúng tiệm cận nhưng **tốn gấp 9 lần** bộ nhớ và thời gian (lưới 300×300 thành 900×900 = 810.000 pixel).
- Ưu điểm: rất khó sai, dễ hình dung; là "đáp án tham chiếu" tốt để đối chiếu các cách nhanh hơn.

Nút thắt: ta phải tạo hẳn lưới lớn chỉ để kiểm tra một điều đơn giản — hai ô kề nhau có cùng mở cửa về phía nhau không. Ta có thể kiểm tra điều đó **trực tiếp**.

## 6. Ý tưởng thuật toán / bước đột phá

**Insight 1 — cạnh hai chiều:** Ô `A` và ô kề `B = A + d` nối nhau **nếu và chỉ nếu** `d ∈ DIRS[A]` **và** `−d ∈ DIRS[B]`. Đồ thị thu được là **vô hướng**: nếu đi được A → B thì đi được B → A.

**Insight 2 — chỉ cần tính liên thông:** Bài toán = "`(0,0)` và `(m−1,n−1)` có cùng thành phần liên thông?" ⇒ BFS/DFS/Union-Find đều được.

**Insight 3 — bậc ≤ 2:** Mỗi ô có đúng 2 cửa ⇒ trong đồ thị nối thật, mỗi đỉnh có bậc ≤ 2. Đồ thị bậc ≤ 2 chỉ gồm **các đường thẳng và chu trình**. Thành phần chứa `(0,0)` là một đường (hoặc vòng) ⇒ ta có thể **đi theo nó** mà không cần `visited`, chỉ nhớ "ô trước đó".

**Quy tắc "nếu… thì…":**
- Nếu ô kế nằm ngoài lưới ⇒ bỏ qua hướng đó.
- Nếu ô kế không có cửa ngược ⇒ không nối, bỏ qua.
- Nếu ô kế đã thăm ⇒ bỏ qua (tránh vòng lặp vô hạn).
- Nếu lấy ra khỏi hàng đợi đúng ô đích ⇒ trả `True`; hàng đợi rỗng ⇒ `False`.

## 7. Các cách tiếp theo

### Cách 2 — BFS với bảng hướng + kiểm tra nối ngược

**Từng đoạn:**
1. `visited[0][0] = True`, cho `(0,0)` vào hàng đợi.
2. Lấy ô `(r,c)` ra; nếu là đích thì `True`.
3. Với mỗi hướng `(dr,dc)` trong `DIRS[grid[r][c]]`: tính ô kế, kiểm tra trong lưới, chưa thăm, và có cửa ngược `(-dr,-dc)`.
4. Đánh dấu thăm **khi cho vào hàng đợi** (không phải khi lấy ra) để mỗi ô vào hàng đợi tối đa một lần.

In [11]:
def valid_path_bfs(grid):
    m, n = len(grid), len(grid[0])
    visited = [[False] * n for _ in range(m)]
    visited[0][0] = True
    q = deque([(0, 0)])
    while q:
        r, c = q.popleft()
        if r == m - 1 and c == n - 1:
            return True
        for dr, dc in DIRS[grid[r][c]]:
            nr, nc = r + dr, c + dc
            if (0 <= nr < m and 0 <= nc < n and not visited[nr][nc]
                    and (-dr, -dc) in DIRS[grid[nr][nc]]):
                visited[nr][nc] = True
                q.append((nr, nc))
    return False

for grid, want in examples:
    print(valid_path_bfs(grid), "| mong đợi", want)

True | mong đợi True
False | mong đợi False
False | mong đợi False


Bản nộp LeetCode (bảng hướng đặt trong class):

In [12]:
class SolutionBFS:
    DIRS = {1: ((0, -1), (0, 1)), 2: ((-1, 0), (1, 0)), 3: ((0, -1), (1, 0)),
            4: ((0, 1), (1, 0)), 5: ((0, -1), (-1, 0)), 6: ((0, 1), (-1, 0))}

    def hasValidPath(self, grid: List[List[int]]) -> bool:
        m, n = len(grid), len(grid[0])
        visited = [[False] * n for _ in range(m)]
        visited[0][0] = True
        q = deque([(0, 0)])
        while q:
            r, c = q.popleft()
            if r == m - 1 and c == n - 1:
                return True
            for dr, dc in self.DIRS[grid[r][c]]:
                nr, nc = r + dr, c + dc
                if (0 <= nr < m and 0 <= nc < n and not visited[nr][nc]
                        and (-dr, -dc) in self.DIRS[grid[nr][nc]]):
                    visited[nr][nc] = True
                    q.append((nr, nc))
        return False

print([SolutionBFS().hasValidPath(g) for g, _ in examples])

[True, False, False]


Độ phức tạp: mỗi ô vào hàng đợi ≤ 1 lần, mỗi ô xét 2 hướng ⇒ **Time `O(m·n)`**, **Space `O(m·n)`** (mảng `visited` + hàng đợi).

### Cách 3 — DFS bằng stack (không đệ quy)

Giống BFS nhưng thay `deque.popleft()` bằng `list.pop()`. Thứ tự thăm khác nhưng tập ô thăm được **giống hệt** (cùng thành phần liên thông). Viết bằng stack để tránh giới hạn đệ quy của Python (mặc định ~1000 tầng, trong khi đường đi có thể dài tới 90.000 ô). Ở đây ta dùng `set` cho `visited` để thấy một kiểu viết khác.

In [13]:
def valid_path_dfs(grid):
    m, n = len(grid), len(grid[0])
    seen = {(0, 0)}
    stack = [(0, 0)]
    while stack:
        r, c = stack.pop()
        if (r, c) == (m - 1, n - 1):
            return True
        for dr, dc in DIRS[grid[r][c]]:
            nr, nc = r + dr, c + dc
            if (0 <= nr < m and 0 <= nc < n and (nr, nc) not in seen
                    and (-dr, -dc) in DIRS[grid[nr][nc]]):
                seen.add((nr, nc))
                stack.append((nr, nc))
    return False

for grid, want in examples:
    print(valid_path_dfs(grid), "| mong đợi", want)

True | mong đợi True
False | mong đợi False
False | mong đợi False


Time `O(m·n)`, Space `O(m·n)`.

### Cách 4 — Union-Find

**Ý tưởng:** duyệt mọi ô; với mỗi ô chỉ xét hai hướng **phải** và **xuống** (để mỗi cặp kề được xét đúng một lần). Nếu ô hiện tại có cửa phải và ô bên phải có cửa trái ⇒ `union`. Tương tự cho xuống/lên. Cuối cùng kiểm tra `find(0) == find(m·n − 1)`.

Ưu điểm: không cần hàng đợi; dễ mở rộng cho nhiều truy vấn "ô A có tới được ô B không?". Nhược: luôn duyệt hết lưới kể cả khi đích ở rất gần.

In [14]:
def valid_path_dsu(grid):
    m, n = len(grid), len(grid[0])
    parent = list(range(m * n))

    def find(x):
        while parent[x] != x:
            parent[x] = parent[parent[x]]
            x = parent[x]
        return x

    def union(a, b):
        ra, rb = find(a), find(b)
        if ra != rb:
            parent[ra] = rb

    for r in range(m):
        for c in range(n):
            t = grid[r][c]
            if c + 1 < n and RIGHT in DIRS[t] and LEFT in DIRS[grid[r][c + 1]]:
                union(r * n + c, r * n + c + 1)
            if r + 1 < m and DOWN in DIRS[t] and UP in DIRS[grid[r + 1][c]]:
                union(r * n + c, (r + 1) * n + c)
    return find(0) == find(m * n - 1)

for grid, want in examples:
    print(valid_path_dsu(grid), "| mong đợi", want)

True | mong đợi True
False | mong đợi False
False | mong đợi False


Bản nộp LeetCode dùng tập số thay cho tuple để kiểm tra nhanh: loại có cửa phải = `{1, 4, 6}`, có cửa trái = `{1, 3, 5}`, có cửa xuống = `{2, 3, 4}`, có cửa lên = `{2, 5, 6}`.

In [15]:
class SolutionDSU:
    def hasValidPath(self, grid: List[List[int]]) -> bool:
        m, n = len(grid), len(grid[0])
        parent = list(range(m * n))

        def find(x):
            while parent[x] != x:
                parent[x] = parent[parent[x]]
                x = parent[x]
            return x

        R, L, D, U = {1, 4, 6}, {1, 3, 5}, {2, 3, 4}, {2, 5, 6}
        for r in range(m):
            for c in range(n):
                t = grid[r][c]
                if c + 1 < n and t in R and grid[r][c + 1] in L:
                    parent[find(r * n + c)] = find(r * n + c + 1)
                if r + 1 < m and t in D and grid[r + 1][c] in U:
                    parent[find(r * n + c)] = find((r + 1) * n + c)
        return find(0) == find(m * n - 1)

print([SolutionDSU().hasValidPath(g) for g, _ in examples])

[True, False, False]


Độ phức tạp: `O(m·n·α(m·n))` ≈ `O(m·n)` thời gian, `O(m·n)` bộ nhớ cho mảng `parent`.

### Cách 5 — Đi theo đường, bộ nhớ `O(1)` (khai thác bậc ≤ 2)

**Ý tưởng:** Ô `(0,0)` có 2 cửa ⇒ có tối đa 2 hướng xuất phát. Thử **từng** hướng. Khi đang ở ô `(r,c)` và vừa đi vào theo hướng `d`, ta biết ta vào từ cửa `−d`; ô này có đúng 2 cửa ⇒ cửa **còn lại** là hướng đi tiếp, không có lựa chọn nào khác. Lặp đến khi:
- tới đích ⇒ `True`;
- ra khỏi lưới hoặc ô kế không có cửa ngược ⇒ hướng này thất bại;
- quay về `(0,0)` ⇒ đi vòng, hướng này thất bại.

**Vì sao không bị lặp vô hạn ở một vòng khác?** Ô đầu tiên bị thăm lại phải là `(0,0)` (chứng minh ở mục 10), và mỗi ô có ≤ 2 hàng xóm nối thật ⇒ sau ≤ `m·n` bước ta chắc chắn dừng.

In [16]:
def valid_path_follow(grid):
    m, n = len(grid), len(grid[0])
    if m == 1 and n == 1:
        return True
    for d0 in DIRS[grid[0][0]]:
        r, c, d = 0, 0, d0
        while True:
            nr, nc = r + d[0], c + d[1]
            back = (-d[0], -d[1])
            if not (0 <= nr < m and 0 <= nc < n) or back not in DIRS[grid[nr][nc]]:
                break                                  # kẹt
            if (nr, nc) == (m - 1, n - 1):
                return True
            if (nr, nc) == (0, 0):
                break                                  # quay về điểm xuất phát
            a, b = DIRS[grid[nr][nc]]
            d = b if a == back else a                  # cửa còn lại
            r, c = nr, nc
    return False

for grid, want in examples:
    print(valid_path_follow(grid), "| mong đợi", want)

True | mong đợi True
False | mong đợi False
False | mong đợi False


Time `O(m·n)` (mỗi hướng đi ≤ `m·n` bước), **Space `O(1)`** — không cần `visited`, không cần hàng đợi. Đây là cách "đẹp" nhất về bộ nhớ, nhưng chỉ dùng được vì mỗi ô có **đúng 2 cửa**; nếu bài có ngã ba/ngã tư thì phải quay về BFS/DFS.

## 8. TRACE từng bước

Trace BFS trên **Ví dụ 1** `[[2,4,3],[6,5,2]]`:

| Bước | Lấy ra | Loại | Hướng thử → ô kế | Nối? | Hàng đợi sau bước |
|---|---|---|---|---|---|
| 1 | (0,0) | 2 `│` | lên → (-1,0) ngoài; xuống → (1,0) loại 6 có lên ✔ | (1,0) | [(1,0)] |
| 2 | (1,0) | 6 `└` | phải → (1,1) loại 5 có trái ✔; lên → (0,0) đã thăm | (1,1) | [(1,1)] |
| 3 | (1,1) | 5 `┘` | trái → (1,0) đã thăm; lên → (0,1) loại 4 có xuống ✔ | (0,1) | [(0,1)] |
| 4 | (0,1) | 4 `┌` | phải → (0,2) loại 3 có trái ✔; xuống → (1,1) đã thăm | (0,2) | [(0,2)] |
| 5 | (0,2) | 3 `┐` | trái → đã thăm; xuống → (1,2) loại 2 có lên ✔ | (1,2) | [(1,2)] |
| 6 | (1,2) | — | là đích | | ⇒ `True` |

In [17]:
NAME = {UP: "lên", DOWN: "xuống", LEFT: "trái", RIGHT: "phải"}

def trace_bfs(grid):
    m, n = len(grid), len(grid[0])
    visited = [[False] * n for _ in range(m)]
    visited[0][0] = True
    q = deque([(0, 0)])
    step = 0
    while q:
        r, c = q.popleft()
        step += 1
        print(f"Bước {step}: lấy ({r},{c}) loại {grid[r][c]} {SHAPE[grid[r][c]]}")
        if (r, c) == (m - 1, n - 1):
            print("  -> tới đích, trả True")
            return True
        for dr, dc in DIRS[grid[r][c]]:
            nr, nc = r + dr, c + dc
            msg = f"  {NAME[(dr, dc)]:5s} -> ({nr},{nc}): "
            if not (0 <= nr < m and 0 <= nc < n):
                print(msg + "ngoài lưới")
            elif visited[nr][nc]:
                print(msg + "đã thăm")
            elif (-dr, -dc) not in DIRS[grid[nr][nc]]:
                print(msg + f"loại {grid[nr][nc]} không có cửa {NAME[(-dr, -dc)]} -> không nối")
            else:
                visited[nr][nc] = True
                q.append((nr, nc))
                print(msg + "nối, thêm vào hàng đợi")
        print("  hàng đợi:", list(q))
    print("  hàng đợi rỗng, trả False")
    return False

trace_bfs([[2, 4, 3], [6, 5, 2]])

Bước 1: lấy (0,0) loại 2 │
  lên   -> (-1,0): ngoài lưới
  xuống -> (1,0): nối, thêm vào hàng đợi
  hàng đợi: [(1, 0)]
Bước 2: lấy (1,0) loại 6 └
  phải  -> (1,1): nối, thêm vào hàng đợi
  lên   -> (0,0): đã thăm
  hàng đợi: [(1, 1)]
Bước 3: lấy (1,1) loại 5 ┘
  trái  -> (1,0): đã thăm
  lên   -> (0,1): nối, thêm vào hàng đợi
  hàng đợi: [(0, 1)]
Bước 4: lấy (0,1) loại 4 ┌
  phải  -> (0,2): nối, thêm vào hàng đợi
  xuống -> (1,1): đã thăm
  hàng đợi: [(0, 2)]
Bước 5: lấy (0,2) loại 3 ┐
  trái  -> (0,1): đã thăm
  xuống -> (1,2): nối, thêm vào hàng đợi
  hàng đợi: [(1, 2)]
Bước 6: lấy (1,2) loại 2 │
  -> tới đích, trả True


True

Trace trên **Ví dụ 3** để thấy chỗ bị kẹt:

In [18]:
trace_bfs([[1, 1, 2]])

Bước 1: lấy (0,0) loại 1 ─
  trái  -> (0,-1): ngoài lưới
  phải  -> (0,1): nối, thêm vào hàng đợi
  hàng đợi: [(0, 1)]
Bước 2: lấy (0,1) loại 1 ─
  trái  -> (0,0): đã thăm
  phải  -> (0,2): loại 2 không có cửa trái -> không nối
  hàng đợi: []
  hàng đợi rỗng, trả False


False

Trace **Cách 5 (đi theo đường)** trên một lưới mà hướng xuất phát đầu tiên đi vòng về `(0,0)`, hướng thứ hai mới dẫn tới đích. Ô `(0,0)` loại 4 `┌` có hai hướng: phải và xuống.

In [19]:
def trace_follow(grid):
    m, n = len(grid), len(grid[0])
    for d0 in DIRS[grid[0][0]]:
        print(f"Thử hướng xuất phát: {NAME[d0]}")
        r, c, d = 0, 0, d0
        while True:
            nr, nc = r + d[0], c + d[1]
            back = (-d[0], -d[1])
            if not (0 <= nr < m and 0 <= nc < n) or back not in DIRS[grid[nr][nc]]:
                print(f"  từ ({r},{c}) đi {NAME[d]} -> ({nr},{nc}) không vào được, bỏ hướng này")
                break
            if (nr, nc) == (m - 1, n - 1):
                print(f"  ({r},{c}) --{NAME[d]}--> ({nr},{nc}) = ĐÍCH -> True")
                return True
            if (nr, nc) == (0, 0):
                print(f"  ({r},{c}) --{NAME[d]}--> (0,0): quay về điểm xuất phát, bỏ hướng này")
                break
            a, b = DIRS[grid[nr][nc]]
            d_next = b if a == back else a
            print(f"  ({r},{c}) --{NAME[d]}--> ({nr},{nc}) loại {grid[nr][nc]} {SHAPE[grid[nr][nc]]}, vào từ {NAME[back]}, ra {NAME[d_next]}")
            r, c, d = nr, nc, d_next
    print("  hết hướng -> False")
    return False

g_loop = [[4, 3, 1], [6, 5, 1], [1, 1, 1]]
show(g_loop)
trace_follow(g_loop)
g_ok = [[4, 1, 3], [2, 1, 2], [6, 1, 5]]
show(g_ok)
trace_follow(g_ok)

  ┌ ┐ ─    [4, 3, 1]
  └ ┘ ─    [6, 5, 1]
  ─ ─ ─    [1, 1, 1]
Thử hướng xuất phát: phải
  (0,0) --phải--> (0,1) loại 3 ┐, vào từ trái, ra xuống
  (0,1) --xuống--> (1,1) loại 5 ┘, vào từ lên, ra trái
  (1,1) --trái--> (1,0) loại 6 └, vào từ phải, ra lên
  (1,0) --lên--> (0,0): quay về điểm xuất phát, bỏ hướng này
Thử hướng xuất phát: xuống
  (0,0) --xuống--> (1,0) loại 6 └, vào từ lên, ra phải
  (1,0) --phải--> (1,1) loại 5 ┘, vào từ trái, ra lên
  (1,1) --lên--> (0,1) loại 3 ┐, vào từ xuống, ra trái
  (0,1) --trái--> (0,0): quay về điểm xuất phát, bỏ hướng này
  hết hướng -> False
  ┌ ─ ┐    [4, 1, 3]
  │ ─ │    [2, 1, 2]
  └ ─ ┘    [6, 1, 5]
Thử hướng xuất phát: phải
  (0,0) --phải--> (0,1) loại 1 ─, vào từ trái, ra phải
  (0,1) --phải--> (0,2) loại 3 ┐, vào từ trái, ra xuống
  (0,2) --xuống--> (1,2) loại 2 │, vào từ lên, ra xuống
  (1,2) --xuống--> (2,2) = ĐÍCH -> True


True

Ở lưới thứ nhất, hướng **phải** đi hết vòng `┌┐┘└` rồi về `(0,0)`; hướng **xuống** cũng chỉ là chiều ngược lại của cùng vòng đó ⇒ `False`. Ở lưới thứ hai, hướng phải đi `─ ┐ │` rồi tới đích `(2,2)` loại 5 `┘` có cửa lên ⇒ `True`.

## 9. Code nộp LeetCode

Chọn **BFS** làm bản chính (rõ ràng, an toàn, `O(m·n)`). Có kèm bản `O(1)` bộ nhớ ở mục 7 (Cách 5) nếu muốn tối ưu.

In [20]:
from collections import deque
from typing import List

class Solution:
    def hasValidPath(self, grid: List[List[int]]) -> bool:
        # hướng (dr, dc) mà mỗi loại đường mở cửa
        dirs = {1: ((0, -1), (0, 1)), 2: ((-1, 0), (1, 0)), 3: ((0, -1), (1, 0)),
                4: ((0, 1), (1, 0)), 5: ((0, -1), (-1, 0)), 6: ((0, 1), (-1, 0))}
        m, n = len(grid), len(grid[0])
        visited = [[False] * n for _ in range(m)]
        visited[0][0] = True
        q = deque([(0, 0)])
        while q:
            r, c = q.popleft()
            if r == m - 1 and c == n - 1:
                return True
            for dr, dc in dirs[grid[r][c]]:
                nr, nc = r + dr, c + dc
                if (0 <= nr < m and 0 <= nc < n and not visited[nr][nc]
                        and (-dr, -dc) in dirs[grid[nr][nc]]):   # ô kế phải nối ngược lại
                    visited[nr][nc] = True
                    q.append((nr, nc))
        return False

for grid, want in examples:
    assert Solution().hasValidPath(grid) == want
print("Solution đúng với 3 ví dụ của đề")

Solution đúng với 3 ví dụ của đề


## 10. Vì sao thuật toán đúng

**Đồ thị đúng nghĩa.** Định nghĩa cạnh giữa ô A và ô kề B = A + d khi `d ∈ DIRS[A]` và `−d ∈ DIRS[B]`. Đường phố thật sự nối A với B đúng khi đoạn đường ở A chạm cạnh chung **và** đoạn đường ở B cũng chạm cạnh chung — chính là hai điều kiện trên. Vậy "có đường hợp lệ" ⇔ "(0,0) và (m−1,n−1) cùng thành phần liên thông".

**BFS/DFS đúng.** Bất biến: mọi ô trong `visited` đều đi tới được từ `(0,0)` (vì chỉ thêm ô kề nối thật của một ô đã tới được). Ngược lại, mọi ô tới được đều sẽ được thêm: giả sử có ô tới được mà không bao giờ được thăm, xét ô đầu tiên như vậy trên một đường đi từ `(0,0)`; ô ngay trước nó đã thăm và khi được lấy ra đã xét cạnh tới nó ⇒ mâu thuẫn.

**Union-Find đúng.** Ta `union` đúng tất cả các cạnh (mỗi cặp kề xét một lần theo hướng phải/xuống, vì cạnh vô hướng). Hai đỉnh cùng gốc ⇔ cùng thành phần liên thông.

**Đi theo đường (Cách 5) đúng.** Mỗi ô có ≤ 2 hàng xóm nối thật. Gọi `P = (0,0), x1, x2, …` là dãy ô khi đi theo một hướng. Giả sử ô đầu tiên bị lặp lại là `xk ≠ (0,0)` ở bước `j > k`. Khi đó `xk` kề với `x(k−1)`, `x(k+1)` và `x(j−1)`; vì `xk` chỉ có 2 hàng xóm, `x(j−1)` trùng `x(k−1)` hoặc `x(k+1)` ⇒ có một ô lặp sớm hơn `xk`, mâu thuẫn. Vậy chỉ có thể quay về `(0,0)` (đi vòng) — khi đó cả thành phần là một chu trình đã đi hết mà không gặp đích. Nếu không vòng, đường dừng ở ngõ cụt; thành phần chứa `(0,0)` là đường thẳng có `(0,0)` ở giữa ⇒ thử **cả hai** hướng là phủ hết thành phần.

## 11. Phân tích Time / Space

Gọi `N = m·n` (số ô, tối đa 90.000).

| Cách | Time | Space | Ghi chú |
|---|---|---|---|
| 1. Phóng to 3×3 | `O(9N)` | `O(9N)` | hằng số lớn |
| 2. BFS | `O(N)` | `O(N)` | mỗi ô vào hàng đợi ≤ 1 lần |
| 3. DFS stack | `O(N)` | `O(N)` | `set` tốn bộ nhớ hơn list 2D |
| 4. Union-Find | `O(N·α(N))` | `O(N)` | luôn duyệt hết lưới |
| 5. Đi theo đường | `O(N)` | `O(1)` | chỉ đúng khi mỗi ô có đúng 2 cửa |

Trường hợp tốt nhất: BFS/DFS/Cách 5 dừng sớm khi `(0,0)` kẹt ngay (`O(1)`). Trường hợp xấu nhất: đường ziczac đi qua mọi ô (`Θ(N)`). Union-Find luôn `Θ(N)`.

## 12. Bộ test

### 12.1 Ví dụ đề + edge case

In [21]:
ALL = {
    "upscale": valid_path_upscale,
    "bfs": valid_path_bfs,
    "bfs_submit": lambda g: SolutionBFS().hasValidPath(g),
    "dfs": valid_path_dfs,
    "dsu": valid_path_dsu,
    "dsu_submit": lambda g: SolutionDSU().hasValidPath(g),
    "follow": valid_path_follow,
    "submit": lambda g: Solution().hasValidPath(g),
}
edge = [
    ([[1]], True),                    # 1 ô: đứng sẵn ở đích
    ([[4]], True),
    ([[1, 1, 1]], True),              # ─ ─ ─
    ([[2], [2], [2]], True),          # │ │ │ theo cột
    ([[1, 2]], False),                # │ không có cửa trái
    ([[2], [1]], False),              # ─ không có cửa trên
    ([[4, 3], [6, 5]], True),         # vòng kín, nhưng đích (1,1) nằm trên vòng
    ([[4, 1], [6, 1]], True),
    ([[6, 1]], True),                 # (0,0) loại 6: cửa phải nối được
    ([[5, 1]], False),                # (0,0) loại 5: cửa trái, trên đều ra ngoài
    ([[1, 1, 1, 1, 1, 1, 3]], True),  # ô cuối ┐ có cửa trái -> vào được
    ([[4, 3, 1], [6, 5, 1]], False),  # vòng 2x2 không thoát ra được
]
for g, want in edge:
    assert valid_path_upscale(g) == want, g      # nhãn kiểm chứng bằng baseline
for name, f in ALL.items():
    for g, want in examples + edge:
        assert f(g) == want, (name, g)
print("Mọi cách đúng với", len(examples + edge), "test")

Mọi cách đúng với 15 test


Hai edge case dễ đoán nhầm: `[[4,3],[6,5]]` là một vòng kín nhưng **đích (1,1) nằm trên vòng** ⇒ `True`; và `─ ─ … ┐` thì ô cuối có cửa trái nên vẫn vào được ⇒ `True`. Vì vậy mọi nhãn edge case đều được kiểm chứng lại bằng baseline phóng to 3×3 trước khi dùng.

### 12.2 Đối chiếu ngẫu nhiên (seed cố định)

Lưới ngẫu nhiên hoàn toàn thường kẹt ngay ở `(0,0)`, nên ta trộn thêm lưới **có cài sẵn một đường** (sinh đường ziczac rồi gán loại đường phù hợp) để có nhiều đáp án `True`.

In [22]:
def street_for(d_in, d_out):
    """Loại đường có hai cửa d_in, d_out."""
    for t, ds in DIRS.items():
        if set(ds) == {d_in, d_out}:
            return t

def planted_grid(rng, m, n):
    """Lưới ngẫu nhiên có cài một đường đơn điệu (chỉ phải/xuống) từ (0,0) tới đích."""
    g = [[rng.randint(1, 6) for _ in range(n)] for _ in range(m)]
    moves = [DOWN] * (m - 1) + [RIGHT] * (n - 1)
    rng.shuffle(moves)
    r, c = 0, 0
    prev = rng.choice([LEFT, UP])                 # cửa "vào" giả cho ô đầu
    for mv in moves:
        g[r][c] = street_for(prev, mv)
        r, c = r + mv[0], c + mv[1]
        prev = (-mv[0], -mv[1])
    g[r][c] = street_for(prev, rng.choice([RIGHT, DOWN]))
    if rng.random() < 0.3:                        # thỉnh thoảng phá đường
        g[rng.randrange(m)][rng.randrange(n)] = rng.randint(1, 6)
    return g

rng = random.Random(1391)
cnt_true = 0
for it in range(1500):
    m, n = rng.randint(1, 7), rng.randint(1, 7)
    if it % 2:
        g = planted_grid(rng, m, n)
    else:
        g = [[rng.randint(1, 6) for _ in range(n)] for _ in range(m)]
    want = valid_path_upscale(g)
    cnt_true += want
    for name, f in ALL.items():
        assert f(g) == want, (name, g)
print("1500 lưới ngẫu nhiên: mọi cách khớp baseline; số lưới True =", cnt_true)

1500 lưới ngẫu nhiên: mọi cách khớp baseline; số lưới True = 684


### 12.3 Giới hạn 300×300: đường rắn bò qua mọi ô

Dựng lưới mà đường đi duy nhất **ziczac qua toàn bộ 90.000 ô** — trường hợp xấu nhất cho BFS/DFS/đi theo đường.

In [23]:
def snake(m, n):
    """Đường rắn: hàng chẵn đi sang phải, hàng lẻ đi sang trái (m lẻ để kết thúc ở góc phải)."""
    g = [[1] * n for _ in range(m)]
    for r in range(m):
        if r % 2 == 0:
            if r > 0:
                g[r][0] = 6            # vào từ trên, ra phải
            if r < m - 1:
                g[r][n - 1] = 3        # vào từ trái, ra dưới
        else:
            g[r][n - 1] = 5            # vào từ trên, ra trái
            g[r][0] = 4                # vào từ phải, ra dưới
    return g

for m, n in [(3, 4), (5, 2)]:
    g = snake(m, n); show(g); print(" ->", Solution().hasValidPath(g))

big = snake(299, 300)
for name in ["bfs", "dfs", "dsu", "follow", "submit"]:
    t = time.perf_counter()
    res = ALL[name](big)
    print(f"{name:7s}: {res}  ({time.perf_counter() - t:.3f}s)")

  ─ ─ ─ ┐    [1, 1, 1, 3]
  ┌ ─ ─ ┘    [4, 1, 1, 5]
  └ ─ ─ ─    [6, 1, 1, 1]
 -> True
  ─ ┐    [1, 3]
  ┌ ┘    [4, 5]
  └ ┐    [6, 3]
  ┌ ┘    [4, 5]
  └ ─    [6, 1]
 -> True


bfs    : True  (0.036s)


dfs    : True  (0.070s)
dsu    : True  (0.037s)
follow : True  (0.024s)


submit : True  (0.036s)


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Quên kiểm tra ô kế có nối ngược lại

Chỉ nhìn cửa của ô hiện tại, rồi "nhảy" sang ô kế bất kể ô đó có mở cửa về phía mình hay không.

In [24]:
def wrong_no_back_check(grid):
    m, n = len(grid), len(grid[0])
    seen = {(0, 0)}
    q = deque([(0, 0)])
    while q:
        r, c = q.popleft()
        if (r, c) == (m - 1, n - 1):
            return True
        for dr, dc in DIRS[grid[r][c]]:
            nr, nc = r + dr, c + dc
            if 0 <= nr < m and 0 <= nc < n and (nr, nc) not in seen:   # SAI: thiếu kiểm tra cửa ngược
                seen.add((nr, nc))
                q.append((nr, nc))
    return False

for g in [[[1, 2]], [[1, 1, 2]], [[2], [1]]]:
    print(g, "| sai:", wrong_no_back_check(g), "| đúng:", Solution().hasValidPath(g))

[[1, 2]] | sai: True | đúng: False
[[1, 1, 2]] | sai: True | đúng: False
[[2], [1]] | sai: True | đúng: False


`[[1,2]]` = `─ │`: ô `(0,0)` có cửa phải, nhưng `│` không có cửa trái ⇒ không đi được. Bản sai vẫn "bước" vào ⇒ trả `True`. Đây cũng chính là Ví dụ 3 của đề.

**Sửa:** thêm `(-dr, -dc) in DIRS[grid[nr][nc]]`.

### Lỗi 2 — Quên `visited` ⇒ lặp vô hạn trên vòng

Với lưới có vòng (ví dụ `┌┐ / └┘`), không đánh dấu thăm sẽ đi vòng mãi. Để notebook không treo, bản minh hoạ dưới đây đếm số bước và **tự dừng sau 1000 bước**.

In [25]:
def wrong_no_visited(grid, limit=1000):
    m, n = len(grid), len(grid[0])
    q = deque([(0, 0)])
    steps = 0
    while q:
        steps += 1
        if steps > limit:
            return f"đã chạy {limit} bước vẫn chưa xong (lặp vô hạn)"
        r, c = q.popleft()
        if (r, c) == (m - 1, n - 1):
            return True
        for dr, dc in DIRS[grid[r][c]]:
            nr, nc = r + dr, c + dc
            if 0 <= nr < m and 0 <= nc < n and (-dr, -dc) in DIRS[grid[nr][nc]]:
                q.append((nr, nc))                     # SAI: không đánh dấu thăm
    return False

g = [[4, 3, 1], [6, 5, 1]]    # vòng 2x2, đích (1,2) không nối với vòng
show(g)
print("sai :", wrong_no_visited(g))
print("đúng:", Solution().hasValidPath(g))

  ┌ ┐ ─    [4, 3, 1]
  └ ┘ ─    [6, 5, 1]
sai : đã chạy 1000 bước vẫn chưa xong (lặp vô hạn)
đúng: False


**Sửa:** đánh dấu `visited` **ngay khi cho vào hàng đợi**.

### Lỗi 3 — DFS đệ quy trên lưới lớn ⇒ `RecursionError`

Đường rắn dài 90.000 ô ⇒ đệ quy sâu 90.000 tầng, vượt giới hạn mặc định của Python (~1000). Ta bắt lỗi để minh hoạ (không để exception văng ra).

In [26]:
def wrong_recursive_dfs(grid):
    m, n = len(grid), len(grid[0])
    seen = set()

    def dfs(r, c):
        if (r, c) == (m - 1, n - 1):
            return True
        seen.add((r, c))
        for dr, dc in DIRS[grid[r][c]]:
            nr, nc = r + dr, c + dc
            if (0 <= nr < m and 0 <= nc < n and (nr, nc) not in seen
                    and (-dr, -dc) in DIRS[grid[nr][nc]]):
                if dfs(nr, nc):
                    return True
        return False

    return dfs(0, 0)

print("lưới nhỏ:", wrong_recursive_dfs(snake(5, 6)))
try:
    print(wrong_recursive_dfs(snake(101, 100)))
except RecursionError as e:
    print("lưới 101x100 -> RecursionError:", str(e)[:50])
print("bản stack/BFS:", valid_path_dfs(snake(101, 100)), Solution().hasValidPath(snake(101, 100)))

lưới nhỏ: True
lưới 101x100 -> RecursionError: maximum recursion depth exceeded
bản stack/BFS: True True


**Sửa:** dùng stack/BFS tường minh (Cách 2, 3) hoặc Cách 5. Tăng `sys.setrecursionlimit` là "chữa cháy", vẫn có thể làm sập stack của trình thông dịch.

### Lỗi 4 — Nhầm chiều "lên/xuống" (dùng `r + 1` cho "lên")

Nếu đảo dấu hướng dọc trong bảng (lên = `(1,0)`), thuật toán vẫn chạy nhưng hiểu sai hình dạng đường.

In [27]:
DIRS_WRONG = {1: [LEFT, RIGHT], 2: [(1, 0), (-1, 0)],
              3: [LEFT, (-1, 0)], 4: [RIGHT, (-1, 0)],       # SAI: "xuống" viết thành (-1, 0)
              5: [LEFT, (1, 0)], 6: [RIGHT, (1, 0)]}         # SAI: "lên" viết thành (1, 0)

def wrong_direction(grid):
    m, n = len(grid), len(grid[0])
    seen = {(0, 0)}
    q = deque([(0, 0)])
    while q:
        r, c = q.popleft()
        if (r, c) == (m - 1, n - 1):
            return True
        for dr, dc in DIRS_WRONG[grid[r][c]]:
            nr, nc = r + dr, c + dc
            if (0 <= nr < m and 0 <= nc < n and (nr, nc) not in seen
                    and (-dr, -dc) in DIRS_WRONG[grid[nr][nc]]):
                seen.add((nr, nc)); q.append((nr, nc))
    return False

for g in [[[4, 3], [6, 5]], [[2, 4, 3], [6, 5, 2]], [[6, 5]]]:
    print(g, "| sai:", wrong_direction(g), "| đúng:", Solution().hasValidPath(g))

[[4, 3], [6, 5]] | sai: False | đúng: True
[[2, 4, 3], [6, 5, 2]] | sai: False | đúng: True
[[6, 5]] | sai: True | đúng: True


**Sửa:** nhớ quy ước hàng 0 ở trên cùng: **lên = `(-1, 0)`**, **xuống = `(1, 0)`**. Vẽ thử một ô ra giấy trước khi gõ bảng.

### Lỗi 5 — Cách 5 chỉ thử một hướng xuất phát

Ô `(0,0)` loại 4 `┌` có hai cửa; nếu chỉ thử hướng đầu tiên trong bảng, có thể bỏ lỡ đường thật đi theo hướng còn lại.

In [28]:
def wrong_follow_one_dir(grid):
    m, n = len(grid), len(grid[0])
    if m == 1 and n == 1:
        return True
    r, c, d = 0, 0, DIRS[grid[0][0]][0]          # SAI: chỉ thử hướng đầu tiên
    while True:
        nr, nc = r + d[0], c + d[1]
        back = (-d[0], -d[1])
        if not (0 <= nr < m and 0 <= nc < n) or back not in DIRS[grid[nr][nc]]:
            return False
        if (nr, nc) == (m - 1, n - 1):
            return True
        if (nr, nc) == (0, 0):
            return False
        a, b = DIRS[grid[nr][nc]]
        d = b if a == back else a
        r, c = nr, nc

g = [[4, 2], [2, 2], [6, 5]]     # hướng 'phải' kẹt, hướng 'xuống' tới đích
show(g)
print("sai:", wrong_follow_one_dir(g), "| đúng:", valid_path_follow(g), Solution().hasValidPath(g))

  ┌ │    [4, 2]
  │ │    [2, 2]
  └ ┘    [6, 5]
sai: False | đúng: True True


## 14. Mẹo làm bài nhanh & nhận dạng pattern

- **"Lưới + ô có hình dạng/cửa"** ⇒ dựng đồ thị ngầm: hàng xóm hợp lệ = trong lưới + **thoả điều kiện hai chiều**.
- Gõ bảng hướng một lần, sau đó toàn bộ logic chỉ là BFS chuẩn. Đặt tên `UP/DOWN/LEFT/RIGHT` giúp tránh lỗi dấu.
- Hỏi "có tới được không?" ⇒ BFS/DFS/Union-Find; hỏi "đường ngắn nhất" ⇒ BFS.
- Mỗi đỉnh bậc ≤ 2 ⇒ có thể "đi theo đường" với `O(1)` bộ nhớ.
- Khi bối rối về điều kiện nối, **phóng to lưới** (3×3 pixel) biến bài thành flood fill thuần tuý — dùng làm bản tham chiếu khi test.
- Với Python và lưới lớn: tránh đệ quy sâu; dùng `deque` hoặc `list` làm stack.

## 15. Biến thể của bài

1. **Trả về đường đi** (danh sách ô) thay vì `True/False` ⇒ lưu `parent` khi BFS rồi truy ngược (Bài 4 ở mục 17).
2. **Đường đi ngắn nhất** (số ô) — vì bậc ≤ 2 nên đường là duy nhất trong thành phần đường thẳng; với ngã ba thì BFS.
3. **Thêm loại ngã ba/ngã tư** (ví dụ loại 7 nối cả 4 phía) ⇒ Cách 5 không còn dùng được, BFS/Union-Find vẫn đúng nguyên xi chỉ cần thêm dòng vào bảng.
4. **Được xoay một số ô** ⇒ trạng thái BFS mở rộng, hoặc 0-1 BFS (chi phí xoay) — giống LeetCode 1368 *Minimum Cost to Make at Least One Valid Path in a Grid*.
5. **Đếm số thành phần liên thông** của mạng đường phố ⇒ Union-Find (Bài 5 ở mục 17).
6. Bài họ hàng: LeetCode 959 *Regions Cut By Slashes* (cũng dùng kỹ thuật phóng to lưới 3×3 hoặc Union-Find theo tam giác).

## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Phóng to 3×3 | vẽ mỗi ô thành 3×3 pixel, flood fill | `O(9mn)` | `O(9mn)` | làm bản tham chiếu; bài kiểu 959 |
| 2. BFS | bảng hướng + kiểm tra nối ngược | `O(mn)` | `O(mn)` | lựa chọn mặc định, cũng cho đường ngắn nhất |
| 3. DFS stack | như BFS nhưng dùng stack | `O(mn)` | `O(mn)` | chỉ cần có/không, thích DFS |
| 4. Union-Find | `union` mọi cặp kề nối thật | `O(mn·α)` | `O(mn)` | nhiều truy vấn liên thông, đếm thành phần |
| 5. Đi theo đường | bậc ≤ 2, luôn chỉ có một cửa còn lại | `O(mn)` | `O(1)` | tối ưu bộ nhớ khi mỗi ô đúng 2 cửa |

## 17. Bài tập vận dụng

### Bài 1 (đoán output)
`Solution().hasValidPath([[4, 1], [6, 1]])` trả về gì? Còn `[[3, 1], [6, 1]]`?

### Bài 2 (tính tay)
Với `grid = [[4, 3, 2], [6, 5, 2], [1, 1, 5]]`, có bao nhiêu ô **tới được** từ `(0,0)`? Đáp án `hasValidPath` là gì?

### Bài 3 (điền TODO)
Hoàn thiện `connected(grid, r, c, nr, nc)`: trả `True` nếu ô `(r,c)` và ô **kề** `(nr,nc)` nối nhau bằng đường phố.

### Bài 4 (tự viết lại)
Viết `find_path(grid)` trả về **danh sách ô** của một đường đi hợp lệ từ `(0,0)` tới đích (dùng BFS + mảng cha), hoặc `[]` nếu không có.

### Bài 5 (mini challenge)
Viết `count_components(grid)`: đếm số **thành phần liên thông** của mạng đường phố (hai ô cùng thành phần nếu đi được từ ô này sang ô kia).

In [29]:
# Bài 3 — TODO (chỉ định nghĩa, không gọi)
def connected(grid, r, c, nr, nc):
    dr, dc = nr - r, nc - c
    # TODO: kiểm tra (nr, nc) nằm trong lưới
    # TODO: kiểm tra (dr, dc) thuộc DIRS[grid[r][c]] và (-dr, -dc) thuộc DIRS[grid[nr][nc]]
    pass

# Bài 4 — TODO
def find_path(grid):
    # TODO: BFS từ (0,0), lưu parent[(nr, nc)] = (r, c)
    # TODO: tới đích thì đi ngược parent để dựng đường, đảo ngược lại
    pass

# Bài 5 — TODO
def count_components(grid):
    # TODO: Union-Find trên m*n ô, union các cặp kề nối thật, đếm số gốc
    pass

## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** (1) `(0,0)` loại 4 có cửa phải và xuống. (2) Xuống tới `(1,0)` loại 6 (có lên ✔, ra phải), phải tới `(1,1)` loại 1 có trái ✔ = đích. (3) Lưới thứ hai: `(0,0)` loại 3 `┐` có trái (ngoài lưới) và xuống → `(1,0)` loại 6 có lên ✔ → phải → `(1,1)` ✔ ⇒ cũng `True`.

**Bài 2.** (1) Vẽ: `┌ ┐ │ / └ ┘ │ / ─ ─ ┘`. (2) Bốn ô góc trên trái tạo vòng kín `┌┐└┘`. (3) `(0,1)` loại 3 không có cửa phải ⇒ không sang được `(0,2)`; vòng không thoát ra được ⇒ 4 ô, `False`.

**Bài 3.** (1) `0 <= nr < m and 0 <= nc < n`. (2) `(dr, dc) in DIRS[grid[r][c]]`. (3) `and (-dr, -dc) in DIRS[grid[nr][nc]]`.

**Bài 4.** (1) `parent = {(0,0): None}` vừa làm `visited`. (2) Tới đích: `while cur is not None: path.append(cur); cur = parent[cur]`. (3) `path[::-1]`; kiểm tra mỗi cặp liên tiếp `connected`.

**Bài 5.** (1) Như Cách 4. (2) Sau khi union hết, đếm `x` mà `find(x) == x`. (3) Kiểm chứng bằng BFS: đếm số lần phải khởi động BFS mới.

## 19. Đáp án tham khảo

In [30]:
# Bài 1 & 2 — kiểm tra
assert Solution().hasValidPath([[4, 1], [6, 1]]) is True
assert Solution().hasValidPath([[3, 1], [6, 1]]) is True

def reachable_count(grid):
    m, n = len(grid), len(grid[0])
    seen = {(0, 0)}
    stack = [(0, 0)]
    while stack:
        r, c = stack.pop()
        for nb in neighbors(grid, r, c):
            if nb not in seen:
                seen.add(nb); stack.append(nb)
    return len(seen)

g2 = [[4, 3, 2], [6, 5, 2], [1, 1, 5]]
assert reachable_count(g2) == 4 and Solution().hasValidPath(g2) is False
print("Bài 1, 2: đúng")

Bài 1, 2: đúng


In [31]:
# Bài 3 — đáp án
def connected_sol(grid, r, c, nr, nc):
    m, n = len(grid), len(grid[0])
    dr, dc = nr - r, nc - c
    if not (0 <= nr < m and 0 <= nc < n):
        return False
    return (dr, dc) in DIRS[grid[r][c]] and (-dr, -dc) in DIRS[grid[nr][nc]]

assert connected_sol([[1, 3]], 0, 0, 0, 1) is True
assert connected_sol([[1, 2]], 0, 0, 0, 1) is False
assert connected_sol([[2], [5]], 0, 0, 1, 0) is True
assert connected_sol([[1]], 0, 0, 0, 1) is False
print("Bài 3: đúng")

Bài 3: đúng


In [32]:
# Bài 4 — đáp án
def find_path_sol(grid):
    m, n = len(grid), len(grid[0])
    parent = {(0, 0): None}
    q = deque([(0, 0)])
    while q:
        cur = q.popleft()
        if cur == (m - 1, n - 1):
            path = []
            while cur is not None:
                path.append(cur)
                cur = parent[cur]
            return path[::-1]
        for nb in neighbors(grid, *cur):
            if nb not in parent:
                parent[nb] = cur
                q.append(nb)
    return []

p = find_path_sol([[2, 4, 3], [6, 5, 2]])
print("đường ví dụ 1:", p)
rng = random.Random(4)
for _ in range(500):
    m, n = rng.randint(1, 6), rng.randint(1, 6)
    g = planted_grid(rng, m, n)
    path = find_path_sol(g)
    assert bool(path) == Solution().hasValidPath(g)
    if path:
        assert path[0] == (0, 0) and path[-1] == (m - 1, n - 1)
        assert all(connected_sol(g, *path[i], *path[i + 1]) for i in range(len(path) - 1))
print("Bài 4: đúng")

đường ví dụ 1: [(0, 0), (1, 0), (1, 1), (0, 1), (0, 2), (1, 2)]
Bài 4: đúng


In [33]:
# Bài 5 — đáp án
def count_components_sol(grid):
    m, n = len(grid), len(grid[0])
    parent = list(range(m * n))

    def find(x):
        while parent[x] != x:
            parent[x] = parent[parent[x]]
            x = parent[x]
        return x

    for r in range(m):
        for c in range(n):
            for nr, nc in ((r, c + 1), (r + 1, c)):
                if connected_sol(grid, r, c, nr, nc):
                    parent[find(r * n + c)] = find(nr * n + nc)
    return sum(1 for x in range(m * n) if find(x) == x)

def count_components_bfs(grid):
    m, n = len(grid), len(grid[0])
    seen, comps = set(), 0
    for r in range(m):
        for c in range(n):
            if (r, c) in seen:
                continue
            comps += 1
            seen.add((r, c)); stack = [(r, c)]
            while stack:
                cr, cc = stack.pop()
                for nb in neighbors(grid, cr, cc):
                    if nb not in seen:
                        seen.add(nb); stack.append(nb)
    return comps

assert count_components_sol([[2, 4, 3], [6, 5, 2]]) == 1
assert count_components_sol([[1, 2, 1], [1, 2, 1]]) == 5
rng = random.Random(5)
for _ in range(500):
    m, n = rng.randint(1, 6), rng.randint(1, 6)
    g = [[rng.randint(1, 6) for _ in range(n)] for _ in range(m)]
    assert count_components_sol(g) == count_components_bfs(g)
print("ví dụ 2 có", count_components_sol([[1, 2, 1], [1, 2, 1]]), "thành phần")
print("Bài 5: đúng")

ví dụ 2 có 5 thành phần
Bài 5: đúng


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template BFS trên lưới có điều kiện nối hai chiều:**

```python
dirs = {loai: ((dr1, dc1), (dr2, dc2)), ...}   # lên=(-1,0), xuống=(1,0)
visited = [[False] * n for _ in range(m)]
visited[0][0] = True
q = deque([(0, 0)])
while q:
    r, c = q.popleft()
    if (r, c) == dich: return True
    for dr, dc in dirs[grid[r][c]]:
        nr, nc = r + dr, c + dc
        if 0 <= nr < m and 0 <= nc < n and not visited[nr][nc] \
           and (-dr, -dc) in dirs[grid[nr][nc]]:      # nối ngược!
            visited[nr][nc] = True
            q.append((nr, nc))
return False
```

**Checklist trước khi submit:**
- [ ] Bảng hướng đúng quy ước: hàng 0 ở trên, **lên = `(-1, 0)`**.
- [ ] Có kiểm tra **ô kế mở cửa ngược lại** `(-dr, -dc)`.
- [ ] Có `visited`, đánh dấu khi cho vào hàng đợi.
- [ ] Kiểm tra biên `0 <= nr < m and 0 <= nc < n` **trước** khi truy cập `grid[nr][nc]`.
- [ ] Lưới 1×1 trả `True`.
- [ ] Không dùng đệ quy sâu (lưới tới 300×300).
- [ ] Đúng chữ ký: `hasValidPath(self, grid: List[List[int]]) -> bool`.